# Contexta Research — 09 Hinglish LID Training

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [2]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = Path("/Users/suvratsmacbookair/MyLocalFiles/NLP project")
os.environ["CONTEXTA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SENSEFORGE_DATA_ROOT"] = str(DATA_ROOT)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Data:", DATA_ROOT)

Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Data: /Users/suvratsmacbookair/MyLocalFiles/NLP project


## Parse human COMI-LINGUA token labels

In [4]:
from senseforge.lid import load_examples,LIDDataset
from senseforge.data import source_paths
paths=source_paths();train=load_examples(paths['lid_train']) if paths['lid_train'].exists() else []
validation=load_examples(paths['lid_test']) if paths['lid_test'].exists() else []
print('Train:',len(train),'Held-out:',len(validation))

Train: 28716 Held-out: 4727


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Training examples:", len(train))
print("Held-out examples:", len(validation))
print("Sample labels:", train[0][1])

PyTorch: 2.15.0
MPS available: True
Training examples: 28716
Held-out examples: 4727
Sample labels: ['hi', 'en', 'en', 'hi', 'hi', 'en', 'en', 'hi', 'hi', 'hi', 'hi', 'en', 'hi', 'en', 'hi', 'hi', 'en', 'en', 'en', 'hi', 'hi']


## Train and save genuine XLM-R token classifier

In [7]:
from sklearn.model_selection import train_test_split
from collections import Counter
import torch

train_data, val_data = train_test_split(
    train,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

test_data = validation

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

BATCH_SIZE = 2
EPOCHS = 3
LEARNING_RATE = 2e-5

print("Device:", device)
print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Batch size:", BATCH_SIZE)

Device: mps
Train: 25844
Validation: 2872
Test: 4727
Batch size: 2


In [8]:
import inspect
from senseforge.lid_training import fit_lid
from senseforge.lid import LIDDataset

print(inspect.signature(fit_lid))
print(inspect.signature(LIDDataset))

(model, train_loader, val_loader, out, epochs=5, patience=2, lr=2e-05)
(examples, tokenizer, label_to_id, max_length=128)


In [9]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.utils.data import DataLoader
from senseforge.lid import LIDDataset
from senseforge.lid_training import fit_lid
from senseforge.training import seed_all

seed_all()

model_name = "xlm-roberta-base"
output_dir = ART / "xlmr_lid_smoke"
output_dir.mkdir(parents=True, exist_ok=True)

labels = sorted({
    tag for _, tags in train_data for tag in tags
})
tag2id = {tag: i for i, tag in enumerate(labels)}

val_clean = [
    (tokens, tags)
    for tokens, tags in val_data
    if set(tags).issubset(tag2id)
]

if not val_clean:
    raise ValueError("No valid validation examples found")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

train_loader = DataLoader(
    LIDDataset(
        train_data[:100],
        tokenizer,
        tag2id,
        max_length=32
    ),
    batch_size=1,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    LIDDataset(
        val_clean[:30],
        tokenizer,
        tag2id,
        max_length=32
    ),
    batch_size=1,
    shuffle=False,
    num_workers=0
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label={i: tag for tag, i in tag2id.items()},
    label2id=tag2id,
    attn_implementation="eager"
)

for param in model.base_model.parameters():
    param.requires_grad = False

gc.collect()

print("Model:", model_name)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Training samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Batch size:", 1)
print("Encoder: Frozen")
print("Attention: Eager")

history = fit_lid(
    model,
    train_loader,
    val_loader,
    output_dir,
    epochs=1,
    patience=1,
    lr=2e-5
)

tokenizer.save_pretrained(output_dir)

print("Smoke test completed")
print("Training history:", history)
print("Checkpoint directory:", output_dir)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: xlm-roberta-base
Trainable parameters: 4614
Training samples: 100
Validation samples: 30
Batch size: 1
Encoder: Frozen
Attention: Eager
Smoke test completed
Training history: [{'epoch': 1, 'train_loss': 2.010593881607056, 'val_loss': 2.0011311928431192, 'accuracy': 0.1079429735234216, 'macro_f1': 0.06800962583449775}]
Checkpoint directory: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/xlmr_lid_smoke


In [17]:
from pathlib import Path

checkpoint_dir = ART / "xlmr_lid_smoke"

for file in sorted(checkpoint_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(checkpoint_dir), f"{file.stat().st_size / 1024:.1f} KB")

config.json 0.9 KB
history.json 0.2 KB
model.safetensors 1083842.5 KB
special_tokens_map.json 0.3 KB
tokenizer.json 16682.6 KB
tokenizer_config.json 1.2 KB


In [19]:
from collections import Counter

train_counts = Counter(
    tag for _, tags in train_data for tag in tags
)

val_counts = Counter(
    tag for _, tags in val_data for tag in tags
)

print("Training labels:")
for tag, count in train_counts.most_common():
    print(f"{tag}: {count:,}")

print("\nValidation labels:")
for tag, count in val_counts.most_common():
    print(f"{tag}: {count:,}")

print("\nNumber of classes:", len(train_counts))

Training labels:
hi: 235,519
h: 111,167
en: 80,683
e: 35,143
ot: 29,587
u: 11,869

Validation labels:
hi: 25,783
h: 12,482
en: 8,938
e: 4,054
ot: 3,211
u: 1,343

Number of classes: 6


In [21]:
sample = train_data[0]

tokens, tags = sample

print("Token count:", len(tokens))
print("Label count:", len(tags))
print("Aligned:", len(tokens) == len(tags))

print("\nFirst 20 token-label pairs:")
for token, tag in list(zip(tokens, tags))[:20]:
    print(f"{token:20} {tag}")

print("\nUnique labels:", sorted(set(
    tag for _, labels in train_data for tag in labels
)))

Token count: 21
Label count: 21
Aligned: True

First 20 token-label pairs:
एसोसिएशन             e
फॉर                  e
डेमोक्रेटिक          e
रिफॉर्म्स            e
(एडीआर)              e
और                   h
कॉमन                 e
कॉज                  e
द्वारा               h
दायर                 h
की                   h
गई                   h
इस                   h
याचिका               h
पर                   h
अब                   h
फरवरी                h
2020                 e
में                  h
सुनवाई               h

Unique labels: ['e', 'en', 'h', 'hi', 'ot', 'u']


In [23]:
from collections import defaultdict

examples = defaultdict(list)

for tokens, tags in train_data:
    for token, tag in zip(tokens, tags):
        if len(examples[tag]) < 12:
            examples[tag].append(token)

for tag in sorted(examples):
    print(f"\n{tag}:")
    print(", ".join(examples[tag]))


e:
एसोसिएशन, फॉर, डेमोक्रेटिक, रिफॉर्म्स, (एडीआर), कॉमन, कॉज, 2020, येलो, अलर्ट, (Yellow, Alert)

en:
ProWatch, X, स्मार्टवॉच, लॉन्च, फीचर्स, JioHotstar, स्ट्रीमिंग, प्लेटफॉर्म, लॉन्च, 50, सब्सक्रिप्शन, Apple

h:
और, द्वारा, दायर, की, गई, इस, याचिका, पर, अब, फरवरी, में, सुनवाई

hi:
भारत, में, लुक, और, हैं, दमदार, रुपये, से, कम, में, पाएं, तीन

ot:
,, ,, ,, !, :, @nycgirl_nikki23, (, @PMOIndia, 3G, 4G, 2G, .

u:
12, ,, ,, 907, 57, 10, 2021, ,, ,, ?, #Kohli, #IndiaChokers


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib

X_train, y_train = [], []
X_val, y_val = [], []

for tokens, tags in train_data:
    X_train.extend(tokens)
    y_train.extend(tags)

for tokens, tags in val_data:
    X_val.extend(tokens)
    y_val.extend(tags)

lid_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=100000
    )),
    ("classifier", SGDClassifier(
        loss="log_loss",
        random_state=42,
        max_iter=1000,
        tol=1e-3
    ))
])

lid_baseline.fit(X_train, y_train)

predictions = lid_baseline.predict(X_val)

print("Validation accuracy:", accuracy_score(y_val, predictions))
print("Validation Macro-F1:", f1_score(y_val, predictions, average="macro"))
print(classification_report(y_val, predictions, zero_division=0))

output_path = ART / "lid_tfidf_sgd.joblib"
joblib.dump(lid_baseline, output_path)

print("Model saved:", output_path)

Validation accuracy: 0.5812832595724857
Validation Macro-F1: 0.2930466216406737
              precision    recall  f1-score   support

           e       0.64      0.06      0.10      4054
          en       0.64      0.73      0.68      8938
           h       0.49      0.00      0.01     12482
          hi       0.57      0.98      0.72     25783
          ot       0.63      0.15      0.24      3211
           u       0.60      0.01      0.01      1343

    accuracy                           0.58     55811
   macro avg       0.59      0.32      0.29     55811
weighted avg       0.57      0.58      0.46     55811

Model saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/lid_tfidf_sgd.joblib


In [27]:
from collections import defaultdict, Counter

token_labels = defaultdict(Counter)

for tokens, tags in train_data:
    for token, tag in zip(tokens, tags):
        token_labels[token.casefold()][tag] += 1

conflicts = []

for token, counts in token_labels.items():
    if len(counts) > 1 and sum(counts.values()) >= 10:
        conflicts.append((
            token,
            dict(counts),
            sum(counts.values())
        ))

conflicts.sort(key=lambda x: x[2], reverse=True)

print("Tokens with conflicting labels:", len(conflicts))

for token, labels, total in conflicts[:25]:
    print(f"{token:20} {labels} Total: {total}")

Tokens with conflicting labels: 5629
के                   {'hi': 9916, 'h': 5518} Total: 15434
में                  {'h': 4046, 'hi': 7458} Total: 11504
की                   {'h': 3022, 'hi': 5562, 'en': 3} Total: 8587
को                   {'h': 2436, 'hi': 4602} Total: 7038
,                    {'ot': 4447, 'u': 2416, 'e': 1, 'hi': 1} Total: 6865
है।                  {'h': 2331, 'hi': 4131} Total: 6462
और                   {'h': 2151, 'hi': 3893} Total: 6044
ने                   {'hi': 3893, 'h': 1864} Total: 5757
से                   {'hi': 3437, 'h': 1956} Total: 5393
.                    {'ot': 3489, 'u': 1802, 'hi': 1} Total: 5292
का                   {'hi': 3249, 'h': 1747} Total: 4996
पर                   {'h': 1209, 'hi': 3056} Total: 4265
hai                  {'hi': 2366, 'h': 391, 'en': 1} Total: 2758
:                    {'ot': 1933, 'u': 749, 'en': 1} Total: 2683
लिए                  {'hi': 1726, 'h': 874} Total: 2600
हैं।                 {'hi': 1393, 'h': 976} Total: 2369


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import joblib

label_map = {
    "h": "hi",
    "hi": "hi",
    "e": "en",
    "en": "en",
    "ot": "other",
    "u": "other"
}

def prepare_lid(examples):
    words, labels = [], []
    for tokens, tags in examples:
        for token, tag in zip(tokens, tags):
            if tag in label_map:
                words.append(token)
                labels.append(label_map[tag])
    return words, labels

X_train_norm, y_train_norm = prepare_lid(train_data)
X_val_norm, y_val_norm = prepare_lid(val_data)

print("Training distribution:", Counter(y_train_norm))
print("Validation distribution:", Counter(y_val_norm))

normalized_lid = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=100000
    )),
    ("classifier", SGDClassifier(
        loss="log_loss",
        random_state=42,
        max_iter=1000,
        tol=1e-3
    ))
])

normalized_lid.fit(X_train_norm, y_train_norm)

pred = normalized_lid.predict(X_val_norm)

print("\nValidation accuracy:", accuracy_score(y_val_norm, pred))
print("Validation Macro-F1:", f1_score(y_val_norm, pred, average="macro"))
print(classification_report(y_val_norm, pred, zero_division=0))

joblib.dump(
    normalized_lid,
    ART / "lid_normalized_experimental.joblib"
)

print("Saved experimental normalized LID model")

Training distribution: Counter({'hi': 346686, 'en': 115826, 'other': 41456})
Validation distribution: Counter({'hi': 38265, 'en': 12992, 'other': 4554})

Validation accuracy: 0.8555302718102166
Validation Macro-F1: 0.6567167622235635
              precision    recall  f1-score   support

          en       0.85      0.75      0.80     12992
          hi       0.86      0.98      0.91     38265
       other       0.85      0.15      0.26      4554

    accuracy                           0.86     55811
   macro avg       0.85      0.63      0.66     55811
weighted avg       0.86      0.86      0.83     55811

Saved experimental normalized LID model


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib

balanced_lid = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        min_df=2,
        max_features=150000,
        sublinear_tf=True
    )),
    ("classifier", SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=1500,
        random_state=42
    ))
])

balanced_lid.fit(X_train_norm, y_train_norm)

balanced_pred = balanced_lid.predict(X_val_norm)

print("Validation accuracy:", accuracy_score(y_val_norm, balanced_pred))
print("Validation Macro-F1:", f1_score(y_val_norm, balanced_pred, average="macro"))
print(classification_report(
    y_val_norm,
    balanced_pred,
    zero_division=0
))

joblib.dump(
    balanced_lid,
    ART / "lid_normalized_balanced.joblib"
)

print("Saved balanced LID model")

Validation accuracy: 0.9504936302879361
Validation Macro-F1: 0.9384298129790484
              precision    recall  f1-score   support

          en       0.87      0.94      0.90     12992
          hi       0.98      0.95      0.97     38265
       other       0.94      0.95      0.95      4554

    accuracy                           0.95     55811
   macro avg       0.93      0.95      0.94     55811
weighted avg       0.95      0.95      0.95     55811

Saved balanced LID model


In [33]:
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import Counter

model = joblib.load(ART / "lid_normalized_balanced.joblib")

X_test, y_test = prepare_lid(test_data)
test_predictions = model.predict(X_test)

print("Test tokens:", len(y_test))
print("Test distribution:", Counter(y_test))

print("\nHeld-out Test Accuracy:", accuracy_score(y_test, test_predictions))
print("Held-out Test Macro-F1:", f1_score(y_test, test_predictions, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, test_predictions, zero_division=0))

print("\nConfusion Matrix (en, hi, other):")
print(confusion_matrix(
    y_test,
    test_predictions,
    labels=["en", "hi", "other"]
))

Test tokens: 100469
Test distribution: Counter({'hi': 66054, 'en': 26505, 'other': 7910})

Held-out Test Accuracy: 0.9606843902099155
Held-out Test Macro-F1: 0.9574023812956671

Classification Report:
              precision    recall  f1-score   support

          en       0.90      0.96      0.93     26505
          hi       0.98      0.96      0.97     66054
       other       0.99      0.95      0.97      7910

    accuracy                           0.96    100469
   macro avg       0.96      0.96      0.96    100469
weighted avg       0.96      0.96      0.96    100469


Confusion Matrix (en, hi, other):
[[25359  1079    67]
 [ 2406 63617    31]
 [  291    76  7543]]
